# 🎙️ PHILIA — Audio Emotion Fine-Tuning
**Base model:** `ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition` (pretrained on RAVDESS) → fine-tuned on MELD  
**Classes:** angry · disgust · fear · happy · neutral · sad · surprise  
**Loss:** Focal Loss + class weights (handles MELD class imbalance)  

## Key features
- **Pretrained emotion encoder** — starts from a model already trained on acted speech (RAVDESS), so it already knows emotion-specific prosody
- **MELD fine-tuning** — adapts to conversational TV-show speech (Friends)
- **Focal Loss + Class Weights** — MELD is heavily imbalanced (neutral ~47%)
- **Early Stopping** — patience=3 epochs to prevent overfitting
- **`freeze_feature_encoder()`** — keeps the lower CNN layers frozen; only the transformer + classifier head are trained

> **Windows:** `dataloader_num_workers=0` required — Windows spawn multiprocessing cannot pickle custom Trainer subclasses.


In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
!pip install -q transformers datasets accelerate evaluate soundfile librosa

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PHILIA/models/audio_emotion'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Model will be saved to:', SAVE_DIR)

In [ ]:
# ── Cell 4: Download MELD dataset ──────────────────────────────────────────
import os

MELD_DIR = '/content/MELD'
os.makedirs(MELD_DIR, exist_ok=True)

if not os.path.exists('/content/MELD.Raw.tar.gz'):
    print('Downloading MELD raw data...')
    !wget -q --show-progress http://web.eecs.umich.edu/~mihalcea/downloads/MELD.Raw.tar.gz -O /content/MELD.Raw.tar.gz
    print('Extracting main archive...')
    !tar -xzf /content/MELD.Raw.tar.gz -C /content/MELD --strip-components=1
    print('Done.')
else:
    print('MELD already downloaded.')

!ls /content/MELD/

In [ ]:
# ── Cell 4.5: Extract MELD sub-archives (train / dev / test mp4 clips) ───────
# The main archive contains sub-archives that must be extracted separately.
import tarfile, os

MELD_DIR = '/content/MELD'

for split in ['train', 'dev', 'test']:
    archive = os.path.join(MELD_DIR, f'{split}.tar.gz')
    if os.path.exists(archive):
        print(f'Extracting {split}.tar.gz...')
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(path=MELD_DIR, filter='data')
    else:
        print(f'WARNING: {archive} not found — skipping')

print('Done extracting all MELD sub-archives.')
!ls /content/MELD/ | head -n 10

In [ ]:
# ── Cell 5: Extract audio from MP4 clips ──────────────────────────────────────
# MELD comes as .mp4 files; we extract 16kHz mono WAV using ffmpeg
import subprocess, glob, os
from tqdm.auto import tqdm

AUDIO_DIR = '/content/MELD_audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

mp4_files = glob.glob('/content/MELD/**/*.mp4', recursive=True)
print(f'Found {len(mp4_files)} mp4 files')

for mp4 in tqdm(mp4_files, desc='Extracting audio'):
    wav_out = os.path.join(AUDIO_DIR, os.path.basename(mp4).replace('.mp4', '.wav'))
    if not os.path.exists(wav_out):
        subprocess.run(
            ['ffmpeg', '-i', mp4, '-ar', '16000', '-ac', '1', '-loglevel', 'error', wav_out],
            check=False
        )

print(f'Audio files extracted: {len(glob.glob(AUDIO_DIR + "/*.wav"))}')

In [ ]:
# ── Cell 6: Load CSV labels and build dataset ──────────────────────────────────
import pandas as pd
import glob, os

# Download CSV files from MELD GitHub (labels for each utterance)
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv -O /content/train.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv   -O /content/dev.csv
!wget -q https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv  -O /content/test.csv

# MELD emotion → PHILIA canonical label
EMOTION_MAP = {
    'anger':    'angry',
    'disgust':  'disgust',
    'fear':     'fear',
    'joy':      'happy',
    'neutral':  'neutral',
    'sadness':  'sad',
    'surprise': 'surprise',
}
LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

AUDIO_DIR = '/content/MELD_audio'

def load_split(csv_path, split_name):
    df = pd.read_csv(csv_path)
    # MELD filename format: diaX_uttY.mp4
    df['wav_path'] = df.apply(
        lambda r: os.path.join(
            AUDIO_DIR,
            f'dia{r["Dialogue_ID"]}_utt{r["Utterance_ID"]}.wav'
        ), axis=1
    )
    df['canonical'] = df['Emotion'].str.lower().map(EMOTION_MAP)
    df = df[df['canonical'].notna()]           # drop any unmapped
    df = df[df['wav_path'].apply(os.path.exists)]  # only files that extracted
    df['label'] = df['canonical'].map(LABEL2ID)
    print(f'{split_name}: {len(df)} samples | dist: {dict(df["canonical"].value_counts())}')
    return df[['wav_path', 'label', 'canonical']].reset_index(drop=True)

train_df = load_split('/content/train.csv', 'TRAIN')
val_df   = load_split('/content/dev.csv',   'VAL')
test_df  = load_split('/content/test.csv',  'TEST')

In [ ]:
# ── Cell 7: Create HuggingFace Dataset + feature extractor ──────────────────
import torch, librosa
import numpy as np
from datasets import Dataset
from transformers import Wav2Vec2FeatureExtractor

MODEL_CHECKPOINT = 'ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition'
SAMPLE_RATE = 16000
MAX_DURATION = 6.0   # seconds — clips > 6s are truncated

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

def df_to_dataset(df):
    return Dataset.from_pandas(df[['wav_path', 'label']])

def preprocess(batch):
    audio_arrays = []
    for path in batch['wav_path']:
        arr, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True,
                              duration=MAX_DURATION)
        audio_arrays.append(arr)
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        padding=True,
        truncation=True,
        max_length=int(SAMPLE_RATE * MAX_DURATION),
        return_tensors='np',
    )
    return {
        'input_values': inputs.input_values,
        'labels': batch['label'],
    }

print('Preprocessing train...')
train_ds = df_to_dataset(train_df).map(preprocess, batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing val...')
val_ds   = df_to_dataset(val_df).map(preprocess,   batched=True, batch_size=32, remove_columns=['wav_path'])
print('Preprocessing test...')
test_ds  = df_to_dataset(test_df).map(preprocess,  batched=True, batch_size=32, remove_columns=['wav_path'])

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('Datasets ready.')

In [ ]:
# ── Cell 8: Load model ────────────────────────────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

# Freeze feature encoder — only fine-tune the transformer layers
model.freeze_feature_encoder()

total   = sum(p.numel() for p in model.parameters())
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total/1e6:.1f}M | Trainable: {trained/1e6:.1f}M')

In [ ]:
# ── Cell 9: Training ──────────────────────────────────────────────────────────
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Optional, Union
import torch

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

@dataclass
class DataCollatorWithPadding:
    feature_extractor: Wav2Vec2FeatureExtractor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_values = [{'input_values': f['input_values']} for f in features]
        labels = torch.tensor([f['labels'] for f in features], dtype=torch.long)
        batch = self.feature_extractor.pad(input_values, padding=self.padding, return_tensors='pt')
        batch['labels'] = labels
        return batch

data_collator = DataCollatorWithPadding(feature_extractor=feature_extractor)

training_args = TrainingArguments(
    output_dir='/content/wav2vec2_audio_emotion',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    learning_rate=3e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,  # Windows: avoid spawn multiprocessing pickling crash
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── Cell 10: Evaluate on test set ─────────────────────────────────────────────
results = trainer.evaluate(test_ds)
print('Test results:', results)

In [ ]:
# ── Cell 11: Save to Google Drive ─────────────────────────────────────────────
print(f'Saving model to {SAVE_DIR} ...')
trainer.model.save_pretrained(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)
print('Done! Files in Drive:')
import os
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ── Cell 12: (Optional) Push to Hugging Face Hub ──────────────────────────────
# Only run this cell if you have a HuggingFace account
# Replace YOUR_HF_USERNAME and YOUR_HF_TOKEN below

# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# trainer.model.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
# feature_extractor.push_to_hub('YOUR_HF_USERNAME/philia-audio-emotion')
print('Skipped (optional). Uncomment above lines to push to HF Hub.')